<a href="https://colab.research.google.com/github/carlosjimenez88M/ai_engineering_henry/blob/main/02-vector_data_bases/04_batman_vector_db_orchestration/00_clase_de_repaso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Henry AI Engineering Bootcamp — Clase de Repaso: Módulos 01 y 02

Este notebook es para quien ya cursó los módulos 01 y 02 y quiere consolidar los conceptos antes de avanzar a sistemas multi-agente. No es una introducción: asume que ya viste los notebooks originales y que tienes preguntas del tipo "¿para qué sirve esto exactamente?" o "¿cómo encajan todas estas piezas?"

Cada sección arranca con el problema que el concepto resuelve, no con la definición. El código es el mismo que usamos en clase, limpiado y comentado.

---

## Tabla de contenidos

| Sección | Concepto | Módulo |
|---------|----------|--------|
| 1 | Validación de datos con Pydantic | 01 |
| 2 | Prompt Engineering y LCEL | 01 |
| 3 | Razonamiento paso a paso: CoT y ReAct | 01 |
| 4 | Flujos condicionales con LangGraph | 01 |
| 5 | Tokens: el lenguaje que entienden los modelos | 02 |
| 6 | TF-IDF vs Embeddings y similitud semántica | 02 |
| 7 | ChromaDB: tu primera base de datos vectorial | 02 |
| 8 | Recuperación robusta: Chunking, BM25 y Multi-Query | 02 |
| 9 | RAG completo: de la búsqueda a la respuesta | 02 |
| 10 | Memoria conversacional (short-term memory) | Udacity Agentic AI |
| 11 | El ciclo de mejora automática: Evaluador-Optimizador | Udacity Agentic AI |
| 12 | Agentic RAG con herramientas (Batman) | 02 |

---

> **Prerequisito:** Necesitas un archivo `.env` con `OPENAI_API_KEY=sk-...` en la raíz del proyecto. Las celdas que requieren llamadas al modelo fallan de forma informativa si no está configurada.

In [ ]:
# Instalación de dependencias — descomenta si estás en Google Colab o en un entorno nuevo
# %pip install -qU langchain langchain-openai langchain-community langgraph chromadb pydantic tiktoken python-dotenv rank_bm25 scikit-learn transformers sentence-transformers faiss-cpu

In [ ]:
import os
import warnings
import json
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# Imports globales de LangChain — disponibles en todo el notebook
# Se separan aqui para que cualquier celda funcione sin depender del orden de ejecucion
try:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    # llm_mini es la instancia base reutilizada en todo el notebook.
    # ChatOpenAI no valida la API key en la instanciacion: solo falla al invocar.
    llm_mini = ChatOpenAI(model='gpt-4o-mini', temperature=0)
except ImportError as e:
    print(f'[HENRY] Dependencias de LangChain no instaladas: {e}')
    print('  -> Ejecuta la celda de instalacion (celda 2) y reinicia el kernel.')
    llm_mini = None
    ChatOpenAI = None
    OpenAIEmbeddings = None
    ChatPromptTemplate = None
    StrOutputParser = None

if not os.environ.get('OPENAI_API_KEY'):
    print('[HENRY] OPENAI_API_KEY no encontrada.')
    print('  -> Crea un archivo .env en la raiz del proyecto con: OPENAI_API_KEY=sk-...')
    print('  -> Las celdas de inferencia LLM fallaran de forma informativa.')
else:
    print('[HENRY] OPENAI_API_KEY detectada. Todas las celdas estan disponibles.')

---
## Módulo 01 — Sección 1: Validación de datos con Pydantic

> **Notebooks fuente:** `00_python_extra_class/` · `01_intro/01_robustez_y_validacion.ipynb` · `01_intro/01_pydantic_pipeline_ai.ipynb`

Imagina que construiste un agente que sugiere acciones financieras. El modelo devuelve un JSON con un campo `confianza` que debería estar entre 0 y 1. Un día el modelo alucina y devuelve `confianza = 1.5`. Tu código lo acepta, lo pasa a downstream, y termina tomando una decisión financiera basada en un valor imposible. Ese bug no se manifiesta en el momento del error: aparece tres pasos después, en producción, sin stack trace claro.

Pydantic es el guardián en la puerta entre el caos del LLM y el orden de tu aplicación. Defines el contrato de datos una vez (qué campos existen, de qué tipo, con qué restricciones) y Pydantic hace cumplir ese contrato en runtime. Si el modelo devuelve algo fuera de rango, obtienes un `ValidationError` limpio antes de que el dato corrupto llegue más lejos.

En AI Engineering, la diferencia entre una app frágil y una robusta suele estar en este punto: ¿tienes un contrato de datos entre el LLM y tu lógica de negocio?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
import typing

class DecisionDeAgente(BaseModel):
    razonamiento: str = Field(description='Justificacion del agente, max 500 chars', max_length=500)
    herramienta: str = Field(description='El nombre de la Tool que usaremos (ej: BusquedaDeCripto)')
    confianza: float = Field(ge=0.0, le=1.0, description='Que tan seguro esta el modelo de 0 a 1')

# Caso invalido: el LLM alucinó una confianza fuera de rango
print('=== Caso INVALIDO ===')
try:
    llm_output_malo = {
        'razonamiento': 'El usuario busca precio historico, invoco el ledger',
        'herramienta': 'LedgerAPI',
        'confianza': 1.5  # supera el maximo permitido de 1.0
    }
    decision = DecisionDeAgente(**llm_output_malo)
except ValidationError as e:
    print('Error de validacion atrapado antes de llegar a produccion.')
    print('El LLM alucinó una confianza mayor a 1.0. Pydantic lo bloqueó en la puerta.')
    print('Detalles:', e.errors()[0]['msg'])

# Caso valido: salida correcta del LLM
print('\n=== Caso VALIDO ===')
llm_output_bueno = {
    'razonamiento': 'El usuario solicita el precio de BTC en los ultimos 30 dias',
    'herramienta': 'LedgerAPI',
    'confianza': 0.92
}
decision_valida = DecisionDeAgente(**llm_output_bueno)
print(f'Decision validada correctamente: herramienta={decision_valida.herramienta}, confianza={decision_valida.confianza}')

---
## Módulo 01 — Sección 2: Prompt Engineering y LCEL

> **Notebooks fuente:** `01_intro/02_prompting/01-basic_example.ipynb` · `01_intro/03_routing/`

Pedirle algo a un modelo sin ejemplos previos (zero-shot) es como pedirle a alguien que te prepare un café por primera vez: puede que funcione, pero no tienes control sobre el resultado. Few-shot prompting es mostrarle tres ejemplos de exactamente cómo lo quieres antes de pedir. El modelo ajusta su distribución de probabilidad para imitar el patrón que le mostraste.

Para tareas donde la forma del output importa tanto como el contenido (extraer un nombre, clasificar en categorías fijas, generar JSON estructurado), few-shot reduce la varianza del modelo de forma significativa sin fine-tuning.

LCEL (LangChain Expression Language) es el pegamento que une los componentes del pipeline. Piensa en el operador `|` como el pipe de Unix, pero para LLMs: cada componente recibe lo que produce el anterior. El flujo siempre sigue el patrón:

```
ChatPromptTemplate  →  formatea el input como mensajes estructurados
ChatModel           →  infiere y devuelve un AIMessage
OutputParser        →  extrae el string (o JSON) del AIMessage
```

Esta cadena es composable: puedes encadenar múltiples prompts, meter una llamada a herramienta en el medio, o ramificar en dos cadenas distintas.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# temperature=0: respuestas deterministas, útil cuando la forma del output es lo que importa
llm_mini = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Few-Shot Prompting: los pares user/assistant son los ejemplos que condicionan el modelo
prompt_extractor = ChatPromptTemplate.from_messages([
    ('system', 'Extrae SOLAMENTE el nombre del Framework Tecnologico del input.'),
    ('user', 'Acabo de levantar el frontend con Next.js y Vercel.'),
    ('assistant', 'Next.js'),
    ('user', 'Levante grafos estocasticos ciclicos.'),
    ('assistant', 'LangGraph'),
    ('user', '{texto_usuario}')
])

# Pipeline LCEL: el operador | encadena los tres componentes
extractor_chain = prompt_extractor | llm_mini | StrOutputParser()

print('--- Resultado del Few-Shot Pipeline LCEL ---')
try:
    resultado = extractor_chain.invoke({'texto_usuario': 'Estoy indexando con TF-IDF usando scikit-learn.'})
    print(resultado)
except Exception as e:
    print(f'[HENRY] Necesitas OPENAI_API_KEY para correr esta celda: {e}')

---
## Módulo 01 — Sección 3: Razonamiento paso a paso: CoT y ReAct

> **Notebooks fuente:** `01_intro/04_cot/01_zero_shot_cot_recomendador.ipynb` · `01_intro/05_react/`

Cuando calculas 17 x 23 en tu cabeza, nadie lo hace en un solo salto. Primero 10 x 23 = 230, luego 7 x 23 = 161, luego sumas. Un modelo de lenguaje comete más errores matemáticos y lógicos cuando se le pide la respuesta directamente porque su arquitectura no está diseñada para razonar en un único paso.

Chain of Thought (CoT) le dice al modelo que piense en voz alta antes de responder. Al forzar pasos intermedios explícitos, los errores de razonamiento quedan expuestos en la cadena de pensamiento y el modelo los puede corregir antes de comprometerse con una respuesta final.

ReAct extiende CoT combinando razonamiento con acciones externas. El ciclo es:
Thought (razona qué necesitas) -> Action (llama a una herramienta) -> Observation (resultado) -> repite hasta llegar a Final Answer.

La instrucción `response_format={"type": "json_object"}` es la forma de decirle a la API de OpenAI que no improvise el formato: solo puede devolver JSON válido. Sin esto, el modelo podría mezclar texto libre con JSON, rompiendo cualquier parser downstream.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
import json

prompt_cot = ChatPromptTemplate.from_messages([
    ('system', 'Resuelve este problema matematico paso a paso y devuelvelo obligatoriamente '
               'en JSON con "thoughts" (lista de str) y "respuesta_final" (entero).'),
    ('user', '{acertijo}')
])

# bind() sobreescribe parámetros del modelo para esta cadena específica
# response_format fuerza JSON válido en la API de OpenAI
llm_json = llm_mini.bind(response_format={'type': 'json_object'})

cot_chain = prompt_cot | llm_json | JsonOutputParser()

print('--- Output Chain of Thought en formato JSON determinístico ---')
try:
    res_cot = cot_chain.invoke({
        'acertijo': 'Tengo 10 contenedores EC2 activos. Dos fallan, y levanto otros 5 en AutoScaling. '
                    'Cuantos totales limpios y estables me quedan?'
    })
    print(json.dumps(res_cot, indent=2, ensure_ascii=False))
except Exception as e:
    print(f'[HENRY] Necesitas OPENAI_API_KEY para correr esta celda: {e}')

---
## Módulo 01 — Sección 4: Flujos condicionales con LangGraph

> **Notebook fuente:** `01_intro/05_routing/routing_langgraph.ipynb`

LCEL es perfecto para pipelines lineales: un input entra, pasa por una secuencia fija de pasos, y un output sale. Pero hay clases de problemas que LCEL no puede modelar bien.

Imagina que quieres que el sistema decida qué rama de código ejecutar según el resultado de un paso anterior. O que repita un nodo hasta que una condición se cumpla. O que ejecute dos ramas en paralelo y combine los resultados. LCEL no tiene primitivas para eso.

LangGraph resuelve esto con un StateGraph: un grafo dirigido donde cada nodo es una función que recibe el estado actual, lo modifica, y lo pasa al siguiente nodo. La diferencia respecto a LCEL es que las aristas pueden ser condicionales: el sistema elige dinámicamente qué nodo activar basándose en el contenido del estado.

Otra forma de pensarlo: LCEL es una receta de cocina (pasos fijos en orden). LangGraph es un diagrama de flujo (hay bifurcaciones, loops, y condiciones de salida).

El estado es un `TypedDict`: el contrato que define exactamente qué campos viajan por el grafo y de qué tipo son. Cada nodo solo puede leer y escribir en esos campos.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class SystemState(TypedDict):
    usuario_input: str
    ruta_decidida: str
    respuesta_maquina: str

def nodo_clasificador(state: SystemState):
    print('[Grafo] Clasificando solicitud...')
    q = state['usuario_input'].lower()
    # En produccion este nodo invocaría un LLM clasificador;
    # aqui usamos heuristico para no consumir tokens en el repaso
    if 'factura' in q:
        return {'ruta_decidida': 'ventas'}
    return {'ruta_decidida': 'tecnico'}

def nodo_robot_ventas(state: SystemState):
    print('[Grafo] Ejecutando Robot de VENTAS')
    return {'respuesta_maquina': 'Generando tickets de Billing API.'}

def nodo_robot_tecnico(state: SystemState):
    print('[Grafo] Ejecutando Robot de SOPORTE')
    return {'respuesta_maquina': 'Escalando log de errores en Datadog.'}

def decide_next_node(state: SystemState) -> str:
    return state['ruta_decidida']

workflow = StateGraph(SystemState)
workflow.add_node('clasificador', nodo_clasificador)
workflow.add_node('ventas', nodo_robot_ventas)
workflow.add_node('tecnico', nodo_robot_tecnico)
workflow.set_entry_point('clasificador')
workflow.add_conditional_edges(
    'clasificador',
    decide_next_node,
    {'ventas': 'ventas', 'tecnico': 'tecnico'}
)
workflow.add_edge('ventas', END)
workflow.add_edge('tecnico', END)

app_grafo = workflow.compile()

print('\n--- TEST: Solicitud de Cobro ---')
res1 = app_grafo.invoke({'usuario_input': 'Tengo un problema con la factura de este mes'})
print('Respuesta Final:', res1['respuesta_maquina'])

print('\n--- TEST: Falla Tecnica ---')
res2 = app_grafo.invoke({'usuario_input': 'Error 500 al compilar el pipeline.'})
print('Respuesta Final:', res2['respuesta_maquina'])

---
## Módulo 02 — Sección 5: Tokens, el lenguaje que entienden los modelos

> **Notebook fuente:** `01_intro/01_tokens.ipynb`

Antes de hablar de embeddings y búsqueda semántica, hay una pregunta básica que se suele saltar: ¿cómo lee texto un modelo de lenguaje? La respuesta no es "letra por letra" ni "palabra por palabra". Los LLMs procesan tokens, que son fragmentos de texto de tamaño variable.

El algoritmo que convierte texto en tokens se llama BPE (Byte-Pair Encoding). La intuición es simple: imagina que quieres comprimir un texto. Las combinaciones de caracteres más frecuentes se convierten en un solo símbolo. BPE repite este proceso iterativamente hasta construir un vocabulario de ~100,000 tokens. Las palabras comunes del inglés son un solo token; las palabras raras o en otros idiomas se parten en sub-palabras.

Esto tiene consecuencias prácticas directas:

- Los modelos de OpenAI cobran por tokens, no por palabras. Si tu sistema procesa documentos   largos, el costo se acumula rápido.
- El español requiere más tokens por idea que el inglés porque el vocabulario BPE fue entrenado   principalmente con texto en inglés.
- Los límites de contexto (128k tokens en gpt-4o-mini) son límites de tokens, no de palabras.   Necesitas chunking cuando tus documentos los superan.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o-mini')

# Demo 1: como se divide una oracion en tokens
oracion = 'La inteligencia artificial transforma el mundo.'
tokens = enc.encode(oracion)
print(f"Oracion: '{oracion}'")
print(f'Tokens (IDs numericos): {tokens}')
print(f'Tokens (decodificados): {[enc.decode([t]) for t in tokens]}')
print(f'Total de tokens: {len(tokens)}\n')

# Demo 2: comparacion Espanol vs Ingles
# El vocabulario BPE fue entrenado principalmente con texto en ingles,
# por lo que el espanol gasta mas tokens por palabra.
texto_es = 'Los modelos de lenguaje comprenden el contexto semanticamente.'
texto_en = 'Language models understand context semantically.'
tokens_es = enc.encode(texto_es)
tokens_en = enc.encode(texto_en)

print('=== Espanol vs Ingles: costo en tokens ===')
print(f'ES ({len(tokens_es)} tokens): {texto_es}')
print(f'EN ({len(tokens_en)} tokens): {texto_en}')
print(f'El espanol usa {len(tokens_es) - len(tokens_en)} tokens mas para expresar la misma idea.\n')

# Demo 3: costo estimado en USD
# gpt-4o-mini: ~$0.15 por millon de tokens de input (precio referencial 2025)
documento = 'Henry AI Engineering Bootcamp ' * 100
tokens_doc = enc.encode(documento)
costo_estimado = (len(tokens_doc) / 1_000_000) * 0.15
print(f'Documento de {len(tokens_doc)} tokens costaria ~${costo_estimado:.6f} USD en input (gpt-4o-mini)')

---
## Módulo 02 — Sección 6: TF-IDF vs Embeddings y similitud semántica

> **Notebooks fuente:** `01_intro/02_rag_tfidf.ipynb` · `02_databases/01-bases-vectoriales-fundamentos.ipynb`

Imagina que tienes una base de datos con el documento "el felino descansaba plácidamente" y un usuario busca "gato dormido". Si usas búsqueda léxica (TF-IDF, Bag of Words), el sistema busca solapamiento de palabras exactas. "Gato" no aparece en el documento y "dormido" tampoco, así que el resultado es relevancia cero. El sistema falló aunque la idea es la misma.

Los embeddings resuelven esto. Una red Transformer comprime cada oración en un vector de alta dimensionalidad (por ejemplo, 1536 dimensiones para text-embedding-3-small). Oraciones con significado similar terminan cerca en ese espacio vectorial, independientemente de las palabras exactas usadas. La distancia entre vectores se mide con cosine similarity.

Escala de referencia práctica para cosine similarity:

```
0.0  →  sin relación semántica (vectores ortogonales)
0.5  →  relacionados pero distintos
0.7+ →  alta similitud semántica
1.0  →  textos idénticos o casi idénticos
```

Ninguno de los dos métodos es universalmente mejor. TF-IDF gana cuando buscas términos exactos (nombres propios, IDs, SKUs). Los embeddings ganan cuando buscas por concepto o intención. En producción conviene combinar ambos, que es lo que veremos en la sección de BM25 + Multi-Query.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from langchain_openai import OpenAIEmbeddings

# TF-IDF: busqueda lexica por solapamiento de palabras exactas
print('--- Busqueda Lexica TF-IDF ---')
corpus = ['Documentacion de LangChain', 'Tutorial de LangGraph y Orquestacion', 'Guia de Python']
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
print('Vocabulario extraido:', vectorizer.get_feature_names_out())

# Embeddings: busqueda semantica vectorial
print('\n--- Busqueda Semantica Vectorial ---')

def cos_sim(vA, vB):
    """Cosine Similarity: mide el angulo entre dos vectores (escala 0.0 a 1.0)"""
    return np.dot(vA, vB) / (np.linalg.norm(vA) * np.linalg.norm(vB))

try:
    emb_model = OpenAIEmbeddings(model='text-embedding-3-small')
    v_gato = np.array(emb_model.embed_query('El gato esta durmiendo'))
    v_feli = np.array(emb_model.embed_query('Un pequeño felino descansa plenamente'))
    v_carro = np.array(emb_model.embed_query('Un motor a combustion de 4 cilindros'))

    sim_semantica = cos_sim(v_gato, v_feli)
    sim_no_rel = cos_sim(v_gato, v_carro)

    print(f'Cosine Similarity (textos semanticamente similares): {sim_semantica:.4f}')
    print(f'  -> {sim_semantica:.2f} indica alta similitud semantica; 1.0 = identicos')
    print(f'\nCosine Similarity (textos sin relacion): {sim_no_rel:.4f}')
    print(f'  -> {sim_no_rel:.2f} indica baja similitud; 0.0 = vectores ortogonales')
except Exception as e:
    print(f'[HENRY] Necesitas OPENAI_API_KEY para los embeddings: {e}')
    print('  -> Resultado esperado: ~0.72 para textos similares, ~0.25 para textos sin relacion')

---
## Módulo 02 — Sección 7: ChromaDB, tu primera base de datos vectorial

> **Notebook fuente:** `02_databases/01-bases-vectoriales-fundamentos.ipynb`

Hasta ahora calculamos embeddings en memoria y los descartamos al final de la sesión. Eso funciona para demos, pero en un sistema real necesitas que esos vectores persistan entre sesiones: los calculas una vez cuando indexas tus documentos, y los recuperas cada vez que llega una consulta. ChromaDB es el lugar donde esos vectores viven.

ChromaDB es un vector store: una base de datos especializada en almacenar y buscar vectores de alta dimensionalidad de forma eficiente. La interfaz básica tiene cuatro operaciones:

- `collection.add()` para indexar documentos
- `collection.query()` para buscar por texto (ChromaDB calcula el embedding internamente)
- `collection.get()` para recuperar por ID específico
- `collection.delete()` para eliminar documentos

Esta celda usa `chromadb.Client()` en memoria, así que no requiere API key ni disco. En producción usarías `chromadb.PersistentClient(path='./chroma_db')` para que los vectores sobrevivan entre reinicios. En el resto del módulo 02, ChromaDB actúa como el backend del pipeline RAG completo.

In [ ]:
import chromadb

# Client() en memoria: no requiere API key ni disco
# En produccion: chromadb.PersistentClient(path='./chroma_db')
client = chromadb.Client()

# CREATE: crear coleccion y agregar documentos
collection = client.create_collection('henry_docs')

collection.add(
    documents=[
        'El plan de salud de NovaTech Solutions es MediPlus.',
        'Los empleados tienen 22 dias de vacaciones al anio.',
        'El seguro dental cubre hasta $150 USD de copago.',
        'Batman patrulla Gotham City con el Batmovil.',
        'LangGraph permite orquestar grafos de agentes de IA.',
    ],
    ids=['doc1', 'doc2', 'doc3', 'doc4', 'doc5']
)
print(f'Coleccion creada con {collection.count()} documentos')

# QUERY: busqueda semantica por texto
# ChromaDB usa un modelo de embedding local por defecto (no requiere API key)
print("\n[QUERY] Busqueda: '¿Que seguro medico tengo?'")
resultados = collection.query(query_texts=['¿Que seguro medico tengo?'], n_results=2)
for i, doc in enumerate(resultados['documents'][0]):
    print(f'  Resultado {i+1}: {doc}')

# GET: recuperar documento por ID especifico
print('\n[GET] Recuperando doc1 por ID:')
doc_recuperado = collection.get(ids=['doc1'])
print(f"  -> {doc_recuperado['documents'][0]}")

# DELETE: eliminar un documento
collection.delete(ids=['doc4'])
print(f'\n[DELETE] Eliminado doc4. Documentos restantes: {collection.count()}')

# Verificar que doc4 ya no existe
verificacion = collection.get(ids=['doc4'])
print(f"[VERIFY] doc4 tras eliminar: {verificacion['documents']}  (lista vacia = eliminado)")

---
## Módulo 02 — Sección 8: Recuperación robusta: Chunking, BM25 y Multi-Query

> **Notebook fuente:** `03_rag/02-rag-avanzado.ipynb`

El RAG básico que veremos en la siguiente sección tiene un problema de recall: si el usuario escribe "MediPlus" pero el chunk más relevante dice "el plan de salud corporativo", la búsqueda semántica puede devolver un resultado incorrecto porque las representaciones vectoriales de un nombre propio son menos estables que las de conceptos generales.

Para documentos que mezclan nombres propios, IDs, códigos y lenguaje natural, conviene combinar tres técnicas:

Chunking recursivo divide el documento en fragmentos con solapamiento (chunk_overlap) para que ningún concepto quede partido entre dos chunks. RecursiveCharacterTextSplitter intenta dividir por párrafos primero, luego por oraciones, y solo divide por caracteres si no queda otra opción.

BM25 es un algoritmo de búsqueda léxica clásico, más robusto que TF-IDF. Es el complemento ideal para embeddings: donde los embeddings fallan con términos exactos, BM25 acierta.

Multi-Query reformula la pregunta del usuario en varias variantes usando un LLM. Si el usuario escribe "seguro médico", una variante podría ser "plan de salud corporativo" y otra "cobertura médica para empleados". Los resultados de todas las variantes se combinan con Reciprocal Rank Fusion (RRF) para maximizar el recall total.

In [ ]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Texto de ejemplo: politicas de NovaTech Solutions (empresa ficticia del bootcamp Henry)
texto_crudo = (
    'Politicas de NovaTech Solutions. '
    'Todos los empleados de tiempo completo tienen 22 dias de vacaciones al anio. '
    'El plan de salud corporativo es MediPlus. '
    'El seguro dental tiene un copago de $150 USD y $20 en medicamentos. '
    'La sede central esta ubicada en la Avenida Independencia 4500.'
)

# chunk_size=120 y chunk_overlap=20 garantizan que 'MediPlus' quede limpiamente
# en un solo chunk sin partirse entre dos fragmentos
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.create_documents([texto_crudo])

print(f'[Chunking Resultante]: {len(chunks)} trozos')
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: '{c.page_content}'")

# BM25: busqueda lexica exacta
# BM25 es ideal para nombres propios y terminos exactos que los embeddings pueden parafrasear
bm25_searcher = BM25Retriever.from_documents(chunks)
print("\n[BM25 Busqueda Lexica — 'MediPlus']:")
try:
    resultado_bm25 = bm25_searcher.invoke('MediPlus')[0].page_content
    print(f"  Chunk mas relevante: '{resultado_bm25}'")
    assert 'MediPlus' in resultado_bm25, 'BUG: BM25 devolvio un chunk que no contiene MediPlus'
    print('  Correcto: el chunk recuperado contiene el termino buscado')
except Exception as e:
    print(f'  [HENRY] Error en BM25: {e}')

# Multi-Query RAG: el LLM genera variantes de la pregunta para ampliar el vocabulario de busqueda
prompt_mq = ChatPromptTemplate.from_template(
    'Genera 3 versiones alternativas de esta pregunta para maximizar el recall en busqueda RAG.\n'
    'Devuelve SOLO las 3 preguntas numeradas, sin explicacion.\n'
    'Pregunta original: {question}'
)
mq_chain = prompt_mq | llm_mini | StrOutputParser()

print('\n[Multi-Query RAG — Ampliacion de Vocabulario]:')
try:
    queries = mq_chain.invoke({'question': '¿Que plan de salud tengo como empleado?'})
    print(queries)
    print('\n  -> En produccion, cada una de estas 3 queries se ejecutaria en ChromaDB')
    print('  -> Los resultados se combinan con RRF para maximo recall')
except Exception as e:
    print(f'  [HENRY] Necesitas OPENAI_API_KEY para Multi-Query: {e}')

---
## Módulo 02 — Sección 9: RAG completo: de la búsqueda a la respuesta

> **Notebook fuente:** `03_rag/02-rag-avanzado.ipynb`

Hasta aquí solo recuperamos documentos relevantes. Eso es útil para alimentar una interfaz de búsqueda, pero no es un sistema de pregunta-respuesta. El salto conceptual de RAG completo es que tomamos los documentos recuperados y los usamos como contexto para que un LLM genere una respuesta en lenguaje natural.

El pipeline tiene tres etapas que van conectadas en secuencia:

Retrieval: el vector store recibe la pregunta del usuario, calcula su embedding, y devuelve los k documentos más cercanos en el espacio vectorial.

Augmentation: esos documentos se insertan como contexto en el prompt, junto con la pregunta original. El LLM recibe ambos y solo puede responder con lo que hay en el contexto.

Generation: el LLM genera la respuesta final. La instrucción explícita de responder solo con el contexto no es un capricho de diseño: es lo que hace el sistema auditable. Si el modelo responde algo, debe estar en los documentos. Si no está, dice que no lo encontró. Eso elimina las alucinaciones más peligrosas: las que parecen correctas.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough

documentos_novatech = [
    'El plan de salud corporativo de NovaTech Solutions es MediPlus.',
    'Los empleados de tiempo completo tienen 22 dias de vacaciones al anio.',
    'El seguro dental tiene un copago de $150 USD y $20 en medicamentos.',
    'La sede central de NovaTech Solutions esta en la Avenida Independencia 4500.',
    'Los beneficios de transporte incluyen tarjeta mensual de $80 USD.',
]

def format_docs(docs):
    """Concatena el contenido de los documentos recuperados en un solo string de contexto."""
    return '\n\n'.join(doc.page_content for doc in docs)

# La instruccion de responder SOLO con el contexto convierte al LLM en un sistema auditable
rag_prompt = ChatPromptTemplate.from_template("""
Responde la pregunta basandote SOLO en el siguiente contexto.
Si no encuentras la respuesta en el contexto, di exactamente: "No encontre esa informacion en los documentos."

Contexto:
{context}

Pregunta: {question}
""")

print('=== RAG Pipeline Completo: Retrieval + Augmentation + Generation ===\n')

try:
    emb_model_rag = OpenAIEmbeddings(model='text-embedding-3-small')
    vector_store = Chroma.from_texts(documentos_novatech, embedding=emb_model_rag)
    retriever = vector_store.as_retriever(search_kwargs={'k': 2})

    rag_chain = (
        {'context': retriever | format_docs, 'question': RunnablePassthrough()}
        | rag_prompt
        | llm_mini
        | StrOutputParser()
    )

    pregunta = '¿Cuantos dias de vacaciones tengo?'
    respuesta = rag_chain.invoke(pregunta)
    print(f'Pregunta: {pregunta}')
    print(f'Respuesta RAG: {respuesta}')

    pregunta2 = '¿Cual es el salario minimo en NovaTech?'
    respuesta2 = rag_chain.invoke(pregunta2)
    print(f'\nPregunta: {pregunta2}')
    print(f'Respuesta RAG: {respuesta2}')
    print('  -> El LLM reconoce que no tiene esa informacion y no alucina')

except Exception as e:
    print(f'[HENRY] Necesitas OPENAI_API_KEY para el RAG completo: {e}')
    print('\nResultado esperado si tuvieras la clave:')
    print('  -> Pregunta: "¿Cuantos dias de vacaciones tengo?"')
    print('  -> Respuesta: "Los empleados de tiempo completo tienen 22 dias de vacaciones al anio."')
    print('  -> Pregunta: "¿Cual es el salario minimo en NovaTech?"')
    print('  -> Respuesta: "No encontre esa informacion en los documentos."')

---
## Udacity Agentic AI — Sección 10: Memoria conversacional (short-term memory)

> **Fuente:** Udacity Agentic AI Course, módulo de memoria conversacional

Un LLM estándar no recuerda nada entre llamadas. Cada vez que invocas el modelo, empieza desde cero, sin saber qué dijiste en el turno anterior. Eso lo convierte en una herramienta de consulta puntual, no en un asistente real. Si le preguntas "¿qué es un agente?" y luego "¿y cuál es su limitación principal?", el modelo trata la segunda pregunta como si fuera la primera: no tiene contexto de que ya estaban hablando de agentes.

La solución es simple pero efectiva: mantener un historial de la conversación como una lista de mensajes y enviarlo completo en cada turno. El modelo recibe el contexto acumulado de toda la conversación y puede dar respuestas coherentes con lo que se dijo antes.

Este patrón tiene un costo: a medida que la conversación crece, crece también el número de tokens que envías en cada llamada. Hay estrategias para manejar esto (truncar el historial, resumirlo, usar memoria externa) pero para conversaciones cortas y medianas la memoria completa es la solución más simple y confiable.

El ejemplo muestra dos sesiones paralelas con personalidades distintas para hacer evidente que el contexto del sistema (SystemMessage) cambia el comportamiento del modelo, y que la segunda pregunta en cada sesión depende del contexto de la primera.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def chat_con_memoria(llm, historial: list, nuevo_mensaje: str) -> str:
    """Agrega el mensaje al historial, invoca el LLM con contexto completo."""
    historial.append(HumanMessage(content=nuevo_mensaje))
    respuesta = llm.invoke(historial)
    historial.append(AIMessage(content=respuesta.content))
    return respuesta.content

# Dos sesiones independientes con personalidades distintas
sesion_tecnica = [SystemMessage(content='Eres un ingeniero de sistemas conciso y preciso.')]
sesion_creativa = [SystemMessage(content='Eres un escritor de ciencia ficcion entusiasta.')]

try:
    r1 = chat_con_memoria(llm_mini, sesion_tecnica, '¿Que es un agente de IA?')
    print('Respuesta tecnica:', r1)

    r2 = chat_con_memoria(llm_mini, sesion_tecnica, '¿Y cual es su limitacion principal?')
    print('Seguimiento tecnico:', r2)
    # La segunda pregunta tiene contexto de la primera gracias al historial

    r3 = chat_con_memoria(llm_mini, sesion_creativa, '¿Que es un agente de IA?')
    print('\nRespuesta creativa:', r3)
    # Misma pregunta, tono completamente diferente por el SystemMessage

except Exception as e:
    print(f'[HENRY] Necesitas OPENAI_API_KEY para esta celda: {e}')

---
## Udacity Agentic AI — Sección 11: El ciclo de mejora automática (Evaluador-Optimizador)

> **Fuente:** Udacity Agentic AI Course, módulo de patrones de agentes

Supón que construiste un agente que genera resúmenes ejecutivos. ¿Cómo sabes si el resultado es bueno? Podrías revisarlo manualmente, pero eso no escala. El patrón Evaluador-Optimizador automatiza ese proceso de revisión dentro del mismo sistema.

La idea tiene dos partes. Primero, un LLM puede evaluar el output de otro LLM si le das criterios claros de lo que es bueno o malo. El evaluador no necesita saber cómo se generó el texto: solo necesita los criterios y el texto a evaluar. Segundo, la retroalimentación del evaluador puede alimentar al generador para que lo intente de nuevo incorporando las correcciones.

Hay dos decisiones de diseño importantes que merecen atención:

La temperatura del evaluador es 0. El evaluador tiene que ser determinista: si evalúas el mismo texto dos veces, quieres el mismo juicio. El generador usa temperatura 0.4 para tener algo de variación al reintentar, porque si generara exactamente lo mismo, nunca mejoraría.

El límite de reintentos (MAX_REINTENTOS) es obligatorio. Sin él, el loop podría correr indefinidamente si el evaluador es muy exigente o si el generador no puede satisfacer los criterios. Siempre necesitas un techo que limite el gasto de tokens y tiempo.

In [ ]:
MAX_REINTENTOS = 3

def generar(llm, tema: str, retroalimentacion: str = '') -> str:
    prompt = f'Escribe un resumen ejecutivo de 3 oraciones sobre: {tema}.'
    if retroalimentacion:
        prompt += f'\n\nEl intento anterior fue rechazado. Retroalimentacion: {retroalimentacion}\nRevisalo.'
    return llm.invoke(prompt).content

def evaluar(llm, resumen: str) -> str:
    """Retorna 'APROBADO' o la razon de rechazo."""
    prompt = (
        f'Evalua este resumen ejecutivo:\n\n{resumen}\n\n'
        'Criterios: debe ser concreto, evitar lenguaje especulativo como "podria" o "quizas", '
        'y tener exactamente 3 oraciones. '
        'Responde solo "APROBADO" o explica que debe corregirse.'
    )
    return llm.invoke(prompt).content

try:
    # temperature=0.4 en el generador: variacion para explorar mejoras en cada reintento
    # temperature=0.0 en el evaluador: juicios deterministas y consistentes
    llm_generador = ChatOpenAI(model='gpt-4o-mini', temperature=0.4)
    llm_evaluador = ChatOpenAI(model='gpt-4o-mini', temperature=0.0)
    tema = 'el impacto de los agentes de IA en la industria financiera'
    retroalimentacion = ''
    for intento in range(MAX_REINTENTOS):
        resumen = generar(llm_generador, tema, retroalimentacion)
        print(f'\n--- Intento {intento + 1} ---')
        print('Resumen:', resumen)
        evaluacion = evaluar(llm_evaluador, resumen)
        print('Evaluacion:', evaluacion)
        if evaluacion.strip().upper().startswith('APROBADO'):
            print('\nResumen aprobado en el intento', intento + 1)
            break
        retroalimentacion = evaluacion
    else:
        print(f'\nSe alcanzo el limite de {MAX_REINTENTOS} intentos.')
except Exception as e:
    print(f'[HENRY] Necesitas OPENAI_API_KEY para esta celda: {e}')

---
## Módulo 02 — Sección 12: Agentic RAG con herramientas (Batman)

> **Notebook fuente:** `04_batman_vector_db_orchestration/05_agent2agent_roles_router_batman.ipynb`

El pipeline RAG que vimos en la sección 9 es estático: el desarrollador escribe el código que decide cuándo recuperar y qué query usar. Agentic RAG invierte eso: el propio LLM decide si necesita consultar la base de datos, con qué parámetros, y si el resultado es suficiente para responder o si necesita otra consulta.

Esto se logra con Tool Calling. El desarrollador define funciones Python y se las pasa al modelo usando `llm.bind_tools()`. El LLM no ejecuta las funciones: devuelve un mensaje estructurado que describe qué función quiere llamar y con qué argumentos. El código externo ejecuta la función, obtiene el resultado, y lo devuelve al modelo para que genere la respuesta final. Ese ciclo es el ReAct loop en su implementación práctica.

En producción, el agente Batman de los notebooks del bootcamp combina este ciclo con ChromaDB como backend de búsqueda, haciendo que el inventario de la Baticueva sea consultable semánticamente por el agente.

In [ ]:
from langchain_core.tools import tool

@tool
def consultar_arsenal_batman(equipo_solicitado: str) -> str:
    """
    Consulta el sistema local de Baticueva para saber que armamento o vehiculos hay disponibles.
    Simula una consulta a ChromaDB Vectorial con los inventarios de la Baticueva.
    """
    print(f"[SYSTEM] Query simulado sobre ChromaDB con '{equipo_solicitado}'...")
    if 'batmovil' in equipo_solicitado.lower():
        return 'Base de datos indica: El Batmovil esta en mantenimiento preventivo (Rueda Izquierda).'
    return 'Equipamiento disponible en perchas 4 y 5.'

# bind_tools transforma al LLM base en un agente capaz de decidir cuándo usar herramientas
batman_agente = llm_mini.bind_tools([consultar_arsenal_batman])

print('[BATMAN AGENTIC RAG] Ciclo: Thought -> Action -> Observation -> Final Answer')
try:
    # El LLM recibe el mensaje y decide autonomamente si invocar la tool
    mensaje_batman = batman_agente.invoke('Identifica si puedo usar el Batmovil para patrullar hoy.')
    print('\n[THOUGHT -> ACTION] El LLM decidio llamar a:')
    print('  Tool calls:', mensaje_batman.tool_calls)
    print('\n  -> En un agente completo, LangGraph ejecutaria la tool y pasaria la Observation')
    print('  -> El LLM generaria entonces la Final Answer basandose en el resultado')
except Exception as e:
    print(f'[HENRY] Necesitas OPENAI_API_KEY para el Agentic RAG: {e}')
    print("  -> Resultado esperado: tool_calls = [{'name': 'consultar_arsenal_batman', 'args': {'equipo_solicitado': 'Batmovil'}}]")

---
## Qué aprendiste y qué sigue

Este notebook recorrió la progresión completa desde las primitivas más básicas hasta los patrones que definen los sistemas de agentes modernos.

En el módulo 01 viste cómo poner barreras alrededor de la estocasticidad del LLM (Pydantic), cómo controlar la forma del output (Prompt Engineering + LCEL), cómo forzar razonamiento explícito (CoT y ReAct), y cómo modelar flujos que van más allá de una cadena lineal (LangGraph).

En el módulo 02 bajaste un nivel más: entendiste cómo el modelo lee texto (tokens y BPE), cómo se compara significado sin comparar palabras (embeddings y cosine similarity), cómo se persisten y consultan esos vectores (ChromaDB), y cómo se construye un sistema completo de pregunta-respuesta que no alucina (RAG con contexto auditado).

Del Udacity Agentic AI course agregamos dos patrones que aparecen en casi todos los sistemas de agentes reales: la memoria conversacional (sin la cual un agente no puede mantener una conversación coherente) y el ciclo Evaluador-Optimizador (sin el cual un agente no puede mejorar su propio output automáticamente).

Los próximos temas que se construyen sobre estas primitivas son:

- Sistemas multi-agente: coordinación de múltiples agentes especializados donde cada uno   tiene su propio rol, herramientas, y memoria
- Evaluación de RAG: métricas cuantitativas de precisión, recall y faithfulness para saber   si tu sistema recupera lo correcto y no inventa
- Despliegue: FastAPI, Docker y las consideraciones de producción para pipelines de IA

Este notebook está diseñado como referencia permanente. Cuando encuentres un concepto confuso en módulos posteriores, vuelve a la sección correspondiente aquí.